# ERA5 Pipeline — Annotated Walkthrough

A step-by-step learning reference for the ERA5 data pipeline.
Covers real ARCO ERA5 from Google Cloud → local zarr → regrid → Dask stats → PyTorch DataLoader.

**Variables (VPD drivers):**

| Variable | Level | Physical role |
|---|---|---|
| `2m_temperature` | Surface | → e_s(T), saturation vapor pressure |
| `2m_dewpoint_temperature` | Surface | → e_a; VPD = e_s(T) - e_s(Td) |
| `10m_u/v_component_of_wind` | Surface | Surface mixing, evaporative demand |
| `temperature` → `temperature_850` | 850 hPa | Air mass advection → drives T2m |
| `specific_humidity` → `specific_humidity_850` | 850 hPa | Moisture advection → drives Td2m |
| `geopotential` → `geopotential_500` | 500 hPa | Synoptic pattern: ridge → hot/dry, trough → cool/moist |

**Source:** `gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`

In [1]:
from __future__ import annotations
import os, shutil, time, warnings
from typing import Sequence

import numpy as np
import torch
import xarray as xr
import dask
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore', category=UserWarning)

## Config

Keep the first pass small — 2 weeks over CONUS fits comfortably in local dev memory.

In [ ]:
ARCO_ERA5_PATH = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'

# Surface variables — no pressure level, shape (time, lat, lon)
SURFACE_VARS = [
    '2m_temperature',            # T2m → e_s(T)
    '2m_dewpoint_temperature',   # Td2m → e_a;  VPD = e_s(T2m) - e_s(Td2m)
    '10m_u_component_of_wind',   # surface zonal wind
    '10m_v_component_of_wind',   # surface meridional wind
]

# Pressure-level variables: ERA5 name → level to extract (hPa)
# Selecting one level converts (time, level, lat, lon) → (time, lat, lon)
LEVEL_VARS = {
    'temperature': 850,       # T850 — warm/cold air mass advection → drives T2m
    'specific_humidity': 850, # q850 — moisture advection → drives Td2m
    'geopotential': 500,      # Z500 — synoptic pattern; ridge = hot/dry, trough = cool/moist
}

# ARCO download list (multi-level vars before extraction)
VARIABLES = SURFACE_VARS + list(LEVEL_VARS.keys())

# Final 2D variable names written to zarr and used by ERA5Dataset
ML_VARS = SURFACE_VARS + [f'{v}_{lev}' for v, lev in LEVEL_VARS.items()]
# → ['2m_temperature', '2m_dewpoint_temperature', '10m_u_component_of_wind',
#    '10m_v_component_of_wind', 'temperature_850', 'specific_humidity_850', 'geopotential_500']

TIME_START = '2020-06-01'
TIME_END   = '2020-06-05'   # 4 days = 96 hourly timesteps

# Northeastern MN / Duluth–Manitou area (ERA5 uses 0–360 longitude)
LAT_MAX = 48.0
LAT_MIN = 45.0
LON_MIN = 263.0   # ~97°W
LON_MAX = 271.0   # ~89°W

LOCAL_ZARR_PATH   = '../data/era5_subset.zarr'
LOCAL_REGRID_PATH = '../data/era5_subset_1deg.zarr'

os.makedirs('../data', exist_ok=True)

## Dask Setup

On M2/32GB, the **threaded scheduler** is still preferred over `LocalCluster`:
- No subprocess spawning overhead
- No port conflicts
- Threads release the GIL for numpy/zarr I/O so parallelism works
- With 32GB headroom, 8 workers matches M2 performance cores

`'synchronous'` = single thread (good for debugging)  
`'threads'` = thread pool (good for I/O-heavy workloads like zarr reads)

`LocalCluster` is now viable on 32GB if you want the task-graph dashboard at `localhost:8787`, but adds ~500MB overhead and isn't needed here.

In [7]:
dask.config.set(scheduler='threads', num_workers=8)
print('Dask scheduler: threaded (8 workers, M2/32GB)')

Dask scheduler: threaded (8 workers, M2/32GB)


## Step 1: Open & Subset ARCO ERA5

`xr.open_zarr` is **lazy** — nothing is downloaded until `.compute()` is called.

Key points:
- `chunks={'time': 1}` tells Dask to treat each timestep as its own task
- ERA5 latitude is stored **descending** (90 → -90), so `slice(lat_max, lat_min)`
- Rename `latitude/longitude` → `lat/lon` for xESMF compatibility
- **Level extraction:** pressure-level vars (`temperature`, `specific_humidity`, `geopotential`) are stored as (time, level, lat, lon). We `.sel(level=N)` to extract a single level and drop the dimension → all vars end up as (time, lat, lon) so ERA5Dataset can stack them

In [ ]:
print('Opening ARCO ERA5 (lazy — nothing downloaded yet)...')

ds = xr.open_zarr(
    ARCO_ERA5_PATH,
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

print(f'Remote vars available: {list(ds.data_vars)[:8]} ...')

# Subset: variables + region + time window
subset = ds[VARIABLES].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),   # descending → slice(max, min)
    longitude=slice(LON_MIN, LON_MAX),
)

subset = subset.rename({'latitude': 'lat', 'longitude': 'lon'})

# Extract specific pressure levels → rename and drop level dimension
# Before: geopotential shape (time, level, lat, lon)
# After:  geopotential_500 shape (time, lat, lon)  — stackable with surface vars
for var, level in LEVEL_VARS.items():
    subset[f'{var}_{level}'] = subset[var].sel(level=level, drop=True)
subset = subset.drop_vars(list(LEVEL_VARS.keys()))

print(f'Subset dims: {dict(subset.sizes)}')
print(f'Variables: {list(subset.data_vars)}')

# Rough memory estimate
n_cells = subset.sizes['time'] * subset.sizes['lat'] * subset.sizes['lon']
mb = n_cells * len(subset.data_vars) * 4 / 1e6
print(f'Estimated full load size: {mb:.0f} MB across {len(subset.data_vars)} variables')

subset

## Step 2: Write Local Zarr

### Chunking strategy: `{time: 1, lat: -1, lon: -1}`

| Access pattern | Optimal chunks |
|---|---|
| ML training (sequential frames) | `{time: 1, lat: -1, lon: -1}` |
| Time-series analysis | `{time: -1, lat: 10, lon: 10}` |
| Spatial statistics | `{time: 24, lat: -1, lon: -1}` |

With `time: 1`, each chunk is one full spatial snapshot (~0.1 MB for CONUS at 0.25°).
ML DataLoader loads frames sequentially → one chunk per `__getitem__` call.

**Important:** `drop_encoding()` clears inherited remote encodings from ARCO ERA5.
Without this, zarr v3 codec format mismatches cause write errors.

In [11]:
ML_CHUNKS = {'time': 1, 'lat': -1, 'lon': -1}  # -1 = full dimension

# Clear inherited remote encodings — required to avoid zarr v3 codec conflicts
subset_chunked = subset.drop_encoding().chunk(ML_CHUNKS)
for v in list(subset_chunked.data_vars) + list(subset_chunked.coords):
    subset_chunked[v].encoding = {}

if os.path.exists(LOCAL_ZARR_PATH):
    shutil.rmtree(LOCAL_ZARR_PATH)

print(f'Chunk layout: {ML_CHUNKS}')
print(f'Writing to {LOCAL_ZARR_PATH} ...')

t0 = time.time()
subset_chunked.to_zarr(
    LOCAL_ZARR_PATH,
    mode='w',
    consolidated=True,   # writes consolidated metadata — faster remote reads
)
print(f'Done in {time.time() - t0:.1f}s')

# Re-open from local — now fully offline
ds_local = xr.open_zarr(LOCAL_ZARR_PATH, consolidated=True, chunks=ML_CHUNKS)
print(f'Local store dims: {dict(ds_local.sizes)}')
ds_local

Chunk layout: {'time': 1, 'lat': -1, 'lon': -1}
Writing to ../data/era5_subset.zarr ...
Done in 656.4s
Local store dims: {'time': 120, 'lat': 13, 'lon': 249, 'level': 37}


<xarray.Dataset> Size: 238MB
Dimensions:                          (time: 120, lat: 13, lon: 249, level: 37)
Coordinates:
  * lat                              (lat) float32 52B 48.0 47.75 ... 45.25 45.0
  * level                            (level) int64 296B 1 2 3 5 ... 950 975 1000
  * lon                              (lon) float32 996B 234.0 234.2 ... 296.0
  * time                             (time) datetime64[ns] 960B 2020-06-01 .....
Data variables:
    2m_dewpoint_temperature          (time, lat, lon) float32 2MB dask.array<chunksize=(1, 13, 249), meta=np.ndarray>
    2m_temperature                   (time, lat, lon) float32 2MB dask.array<chunksize=(1, 13, 249), meta=np.ndarray>
    geopotential                     (time, level, lat, lon) float32 57MB dask.array<chunksize=(1, 37, 13, 249), meta=np.ndarray>
    leaf_area_index_high_vegetation  (time, lat, lon) float32 2MB dask.array<chunksize=(1, 13, 249), meta=np.ndarray>
    specific_humidity                (time, level, lat, lon) float32 57MB dask.array<chunksize=(1, 37, 13, 249), meta=np.ndarray>
    t2m_celsius                      (time, lat, lon) float32 2MB dask.array<chunksize=(1, 13, 249), meta=np.ndarray>
    u_component_of_wind              (time, level, lat, lon) float32 57MB dask.array<chunksize=(1, 37, 13, 249), meta=np.ndarray>
    v_component_of_wind              (time, level, lat, lon) float32 57MB dask.array<chunksize=(1, 37, 13, 249), meta=np.ndarray>
    volumetric_soil_water_layer_1    (time, lat, lon) float32 2MB dask.array<chunksize=(1, 13, 249), meta=np.ndarray>
Attributes:
    last_updated:           2026-03-15 02:17:18.543343+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-11-30
    valid_time_stop_era5t:  2026-03-09

## Step 3: Regrid with xESMF

### Which method to use?

| Method | Use for | Preserves |
|---|---|---|
| `bilinear` | Temperature, wind, geopotential, soil moisture | Local point values |
| `conservative` | Precipitation, radiation, surface fluxes | Area integrals (mass/energy) |
| `nearest_s2d` | Categorical / mask fields | Discrete categories |

Conservative regridding is physically critical: bilinear-interpolated precipitation
can violate conservation of mass. ERA5 0.25° → WRF boundary conditions always
requires conservative for flux variables.

In [ ]:
try:
    import xesmf as xe

    target_grid = xr.Dataset({
        'lat': (['lat'], np.arange(LAT_MIN, LAT_MAX + 1.0, 1.0)),
        'lon': (['lon'], np.arange(LON_MIN, LON_MAX + 1.0, 1.0)),
    })

    print(f'Source: {ds_local.sizes["lat"]} × {ds_local.sizes["lon"]} (0.25°)')
    print(f'Target: {len(target_grid.lat)} × {len(target_grid.lon)} (1.0°)')

    state_vars = ['2m_temperature', 'volumetric_soil_water_layer_1',
                  'leaf_area_index_high_vegetation', 't2m_celsius']
    state_vars = [v for v in state_vars if v in ds_local]

    regridder = xe.Regridder(
        ds_local[state_vars],
        target_grid,
        method='bilinear',
        periodic=False,
        reuse_weights=True,   # cache weights — saves time if called repeatedly
    )

    ds_regrid = regridder(ds_local[state_vars])
    print(f'Regridded dims: {dict(ds_regrid.sizes)}')

    if os.path.exists(LOCAL_REGRID_PATH):
        shutil.rmtree(LOCAL_REGRID_PATH)

    ds_regrid.chunk({'time': 1, 'lat': -1, 'lon': -1}).to_zarr(
        LOCAL_REGRID_PATH, mode='w', consolidated=True
    )
    print(f'Saved: {LOCAL_REGRID_PATH}')

except ImportError:
    print('xESMF not installed — skipping regrid step.')
    print('Install: conda install -c conda-forge xesmf')

## Step 4: Dask Parallel Stats

### Key concept: lazy evaluation

```python
# Without Dask:
ds = xr.open_dataset('huge_era5.nc')  # loads everything into RAM → crash
mean = ds.t850.mean()                 # computes on full array

# With Dask:
ds = xr.open_zarr('era5.zarr')        # loads nothing — just metadata
mean = ds.t850.mean()                 # builds a task graph — nothing computed yet
result = mean.compute()              # NOW it executes, chunk by chunk
```

Build **all** graphs before calling `.compute()` — Dask can then fuse operations
and avoid reading each chunk more than once.

In [12]:
    # 'geopotential', '2m_temperature','u_component_of_wind',
    # 'v_component_of_wind', 'specific_humidity', '2m_dewpoint_temperature',
    # 'volumetric_soil_water_layer_1',
    # 'leaf_area_index_high_vegetation',

# Build all lazy graphs before computing any
temp_mean = (ds_local['2m_temperature'] - 273.15).mean('time')
temp_std = (ds_local['2m_temperature'] - 273.15).std('time')
temp_max = (ds_local['2m_temperature'] - 273.15).max('time')

geopotential_mean  = ds_local['geopotential'].mean('time')
geopotential_std  = ds_local['geopotential'].std('time')
geopotential_max   = ds_local['geopotential'].max('time')

print('Lazy graphs built — computing now ...')
t0 = time.time()

# Compute all three in one pass — Dask reads each chunk once for all ops
temp_mean_val, temp_std_val, temp_max_val = dask.compute(
    temp_mean, temp_std, temp_max
)
elapsed = time.time() - t0

print(f'Computed in {elapsed:.2f}s')
print(f'Mean 2m temp (°C):          {float(temp_mean_val.mean()):.2f}')
print(f'2m temp std:  {float(temp_std_val.mean()):.4f}')
print(f'2m temp max:            {float(temp_max_val.mean()):.4f}')

Lazy graphs built — computing now ...
Computed in 0.10s
Mean 2m temp (°C):          15.23
2m temp std:  4.5986
2m temp max:            24.9695


## Step 5: PyTorch Dataset / DataLoader

### Design decisions

- `__getitem__` uses `isel(time=slice(...))` — loads only `seq_len+1` chunks, not the full dataset
- Normalization stats computed once at init, stored as scalars. In production: compute over a full year and save to JSON
- `num_workers=4, persistent_workers=True`: M2/32GB handles multiprocess workers without OOM (was 0 on M1 due to fork copy-on-write memory pressure)
- `pin_memory=False`: only helps with CUDA, not MPS

In [13]:
class ERA5Dataset(Dataset):
    """
    Zarr-backed sliding window dataset for ConvLSTM training.

    Returns:
        x: (seq_len, C, H, W)  — input sequence
        y: (C, H, W)           — target next frame
    """

    def __init__(self, zarr_path: str, variables: Sequence[str], seq_len: int = 6) -> None:
        self.ds        = xr.open_zarr(zarr_path, consolidated=True)[list(variables)]
        self.variables = list(variables)
        self.seq_len   = seq_len
        self.n_times   = self.ds.sizes['time']

        print('  Computing normalization stats ...')
        self.means: dict[str, float] = {}
        self.stds:  dict[str, float] = {}

        for v in self.variables:
            mean_val = float(self.ds[v].mean().compute())
            std_val  = float(self.ds[v].std().compute())
            self.means[v] = mean_val
            self.stds[v]  = std_val if std_val > 0 else 1.0
            print(f'    {v}: mean={mean_val:.4f}  std={std_val:.4f}')

    def __len__(self) -> int:
        return self.n_times - self.seq_len

    def __getitem__(self, idx: int):
        # isel with slice loads only these chunks — not the full dataset
        window = self.ds.isel(time=slice(idx, idx + self.seq_len + 1))

        arr = np.stack(
            [((window[v].values - self.means[v]) / self.stds[v]).astype(np.float32)
             for v in self.variables],
            axis=1,
        )  # (T+1, C, H, W)

        x = torch.from_numpy(arr[:self.seq_len])   # (seq_len, C, H, W)
        y = torch.from_numpy(arr[self.seq_len])    # (C, H, W)
        return x, y

In [ ]:
# Debug: check shapes of each variable in a sample window
import xarray as xr
ds_check = xr.open_zarr(LOCAL_ZARR_PATH, consolidated=True)
window = ds_check.isel(time=slice(0, 7))
for v in VARIABLES:
    arr = window[v].values
    print(f'{v}: {arr.shape}')

In [ ]:
import xarray as xr
    ds_check = xr.open_zarr(LOCAL_ZARR_PATH, consolidated=True)
    window = ds_check.isel(time=slice(0, 7))
    for v in VARIABLES:
        arr = window[v].values
        print(f'{v}: {arr.shape}')

IndentationError: unexpected indent (2295867613.py, line 2)

In [17]:
dataset = ERA5Dataset(
    zarr_path=LOCAL_ZARR_PATH,
    variables= ['geopotential', '2m_temperature','u_component_of_wind',
    'v_component_of_wind', 'specific_humidity', '2m_dewpoint_temperature'],
    # ['2m_temperature', 'volumetric_soil_water_layer_1', 'leaf_area_index_high_vegetation'],
    seq_len=6,
)

print(f'Dataset length: {len(dataset)} windows')
x, y = dataset[0]
print(f'Sample x: {x.shape}  (seq_len=6, C=3, H, W)')
print(f'Sample y: {y.shape}  (C=3, H, W)')
print(f'x mean: {x.mean():.4f}  std: {x.std():.4f}  (should be ~0, ~1)')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=False)
batch_x, batch_y = next(iter(loader))
print(f'Batch x: {batch_x.shape}  (B=4, T=6, C=3, H, W)')
print(f'Batch y: {batch_y.shape}  (B=4, C=3, H, W)')

  Computing normalization stats ...
    geopotential: mean=127928.7812  std=131758.2500
    2m_temperature: mean=288.3832  std=6.4887
    u_component_of_wind: mean=7.4094  std=16.4525
    v_component_of_wind: mean=-1.1108  std=9.0129
    specific_humidity: mean=0.0019  std=0.0026
    2m_dewpoint_temperature: mean=279.7976  std=4.6902
Dataset length: 114 windows


ValueError: all input arrays must have the same shape

## Cloud Storage Patterns

The zarr API is identical for local and cloud stores — just swap the path:

```python
# Read public ARCO ERA5 (no credentials)
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

# Write to GCS
ds.to_zarr('gs://your-bucket/era5_processed.zarr', mode='w', consolidated=True)

# Write to S3
import s3fs
store = s3fs.S3Map('s3://your-bucket/era5_processed.zarr', s3=s3fs.S3FileSystem())
ds.to_zarr(store, mode='w', consolidated=True)
```

Point `ERA5Dataset` at a cloud zarr to train directly from cloud storage:

```python
dataset = ERA5Dataset(
    zarr_path='gs://your-bucket/era5_processed.zarr',
    variables=['2m_temperature', ...],
)
```